In [ ]:
import pandas as pd
import numpy as np

import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

import joblib

### 1. Data Loading & Cleaning

In [ ]:
def load_fasttext_data(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(None, 1)
            if len(parts) == 2:
                label, text = parts
                label = label.replace('__label__', '')
                data.append([label, text])
    df = pd.DataFrame(data, columns=['label', 'text'])
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    return df.dropna(subset=['label'])

df1 = load_fasttext_data('Datasets/dataset 1.txt')
df2 = load_fasttext_data('Datasets/dataset 2.txt')
df3 = load_fasttext_data('Datasets/dataset 3.txt')

df4 = pd.read_csv('Datasets/dataset 4.csv')[['class_index', 'review_text']] \
        .rename(columns={'class_index': 'label', 'review_text': 'text'})

df5 = pd.read_csv('Datasets/Neutral Dataset.csv')[['class_index', 'review_text']] \
        .rename(columns={'class_index': 'label', 'review_text': 'text'})

df4['label'] = pd.to_numeric(df4['label'], errors='coerce')
df5['label'] = pd.to_numeric(df5['label'], errors='coerce')

df4 = df4.dropna(subset=['label'])
df5 = df5.dropna(subset=['label'])

combined_dataset = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

combined_dataset['label'] = combined_dataset['label'].astype(int)
combined_dataset = combined_dataset.dropna(subset=['text'])

label_mapping = {
    1: "Negative",
    2: "Neutral",
    3: "Positive"
}

combined_dataset['category'] = combined_dataset['label'].map(label_mapping)
combined_dataset = combined_dataset.dropna(subset=['category'])
combined_dataset = combined_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

X = combined_dataset['text']
y = combined_dataset['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

custom_stop_words = list(
    ENGLISH_STOP_WORDS.difference({'not', 'no', 'very', 'but', 'against'})
)

print("Dataset shape:", combined_dataset.shape)
print("Class distribution:\n", combined_dataset['category'].value_counts())

### 2. Optuna Optimization

In [ ]:
sample_idx = X_train.sample(n=80000, random_state=42).index
X_sample = X_train.loc[sample_idx]
y_sample = y_train.loc[sample_idx]

def objective(trial):
    classifier_name = trial.suggest_categorical(
        'classifier', ['LogisticRegression', 'LinearSVC', 'MultinomialNB']
    )
    
    if classifier_name == 'LogisticRegression':
        clf = LogisticRegression(
            C=trial.suggest_float('lr_c', 0.1, 5.0, log=True),
            solver='saga',
            class_weight='balanced',
            max_iter=2000
        )
        
    elif classifier_name == 'LinearSVC':
        clf = LinearSVC(
            C=trial.suggest_float('svc_c', 0.01, 1.0, log=True),
            class_weight='balanced',
            max_iter=2000
        )
    
    elif classifier_name == 'MultinomialNB':
        clf = MultinomialNB(
            alpha=trial.suggest_float('nb_alpha', 0.01, 10.0, log=True)
        )

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words=custom_stop_words,
            max_features=120000,
            ngram_range=(1, 3),
            sublinear_tf=True
        )),
        ('clf', clf)
    ])

    return cross_val_score(pipeline, X_sample, y_sample, n_jobs=3, cv=3).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)


### 3. Final Stacking Ensemble

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

bp = study.best_params
print(f"\nBest Params Found: {bp}")

estimators = [
    ('lr', LogisticRegression(
        C=bp.get('lr_c', 0.5),
        solver='saga',
        class_weight='balanced',
        max_iter=1000
    )),
    
    ('svc', LinearSVC(
        C=bp.get('svc_c', 0.1),
        class_weight='balanced',
        max_iter=3000
    )),

    ('nb', MultinomialNB(
        alpha=bp.get('nb_alpha', 1.0)
    ))
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=XGBClassifier(
        use_label_encoder=False,
        eval_metric='mlogloss',
        n_jobs=1
    ),cv=3, n_jobs=3
)

final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=custom_stop_words,
        max_features=100000,
        ngram_range=(1, 3),
        sublinear_tf=True
    )),
    ('stacker', stacking_clf)
])

final_pipeline.fit(X_train, y_train)


### 4. Final Training & Prediction

In [ ]:
y_pred = final_pipeline.predict(X_test)
print("\n" + "="*50)
print("FINAL STACKING PERFORMANCE (3 CLASSES - LR+SVC+NB ENSEMBLE)")
print("="*50)
print(classification_report(y_test, y_pred))


In [ ]:
def predict_sentiment(text):
    return final_pipeline.predict([text])[0]

test_sentence = "The student is doing great in studies but the behaviour is not good in class."
print(f'\nTest sentence: "{test_sentence}"')
print(f'Predicted category: {predict_sentiment(test_sentence)}')

### 5. Exporting model

In [ ]:
model_filename = 'sentiment_analysis_model.pkl'
joblib.dump(final_pipeline, model_filename)

### 6. Testing model

In [ ]:
from joblib import load

model = load("sentiment_analysis_model.pkl")

sentence = ["?"]

print(model.predict(sentence))